In [1]:
import os

# A Dockerfile is like a recipe
# It tells Docker exactly how to build your project
# Step by step - install Python, install libraries, copy code, run API

dockerfile = """
FROM python:3.11-slim

# Set working directory inside container
WORKDIR /app

# Copy requirements first (for faster builds)
COPY requirements.txt .

# Install all libraries
RUN pip install --no-cache-dir -r requirements.txt

# Copy the API code
COPY src/serving/api.py .

# Expose port 8000 so outside world can reach the API
EXPOSE 8000

# Command to run when container starts
CMD ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"]
"""

# Save Dockerfile
path = os.path.expanduser("~/churn-mlops/Dockerfile")
with open(path, 'w') as f:
    f.write(dockerfile)

print("✅ Dockerfile created!")
print(f"📁 Saved to: {path}")
print("\nDockerfile contents:")
print(dockerfile)

✅ Dockerfile created!
📁 Saved to: /home/sagemaker-user/churn-mlops/Dockerfile

Dockerfile contents:

FROM python:3.11-slim

# Set working directory inside container
WORKDIR /app

# Copy requirements first (for faster builds)
COPY requirements.txt .

# Install all libraries
RUN pip install --no-cache-dir -r requirements.txt

# Copy the API code
COPY src/serving/api.py .

# Expose port 8000 so outside world can reach the API
EXPOSE 8000

# Command to run when container starts
CMD ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"]



In [2]:
# This file lists every library our project needs
# Docker uses this to install everything inside the container

requirements = """pandas==2.0.3
numpy==1.24.3
scikit-learn==1.3.0
xgboost==2.0.0
optuna==3.3.0
mlflow==2.7.0
fastapi==0.103.1
uvicorn==0.23.2
boto3==1.28.44
shap==0.42.1
pytest==7.4.2
python-dotenv==1.0.0
joblib==1.3.2
pydantic==2.3.0
requests==2.31.0
scipy==1.11.2
"""

path = os.path.expanduser("~/churn-mlops/requirements.txt")
with open(path, 'w') as f:
    f.write(requirements)

print("✅ requirements.txt created!")
print(f"📁 Saved to: {path}")

✅ requirements.txt created!
📁 Saved to: /home/sagemaker-user/churn-mlops/requirements.txt


In [3]:
# Tests check that our code works correctly
# Before deploying, GitHub Actions runs these tests
# If any test fails - deployment stops automatically!
# This prevents broken code from reaching production

test_code = """
import pytest
import numpy as np
import pandas as pd
import joblib
import boto3
import json
import io

BUCKET_NAME = "churn-mlops-project-cherry"

# ── Test 1: Model loads correctly ─────────────────────────
def test_model_loads():
    s3 = boto3.client('s3')
    s3.download_file(
        BUCKET_NAME,
        'models/latest/xgboost_churn_model.pkl',
        '/tmp/test_model.pkl'
    )
    model = joblib.load('/tmp/test_model.pkl')
    assert model is not None
    print("✅ Test 1 passed: Model loads correctly")

# ── Test 2: Scaler loads correctly ────────────────────────
def test_scaler_loads():
    s3 = boto3.client('s3')
    s3.download_file(
        BUCKET_NAME,
        'models/scaler.pkl',
        '/tmp/test_scaler.pkl'
    )
    scaler = joblib.load('/tmp/test_scaler.pkl')
    assert scaler is not None
    print("✅ Test 2 passed: Scaler loads correctly")

# ── Test 3: Model makes predictions ───────────────────────
def test_model_predicts():
    s3 = boto3.client('s3')
    s3.download_file(
        BUCKET_NAME,
        'models/latest/xgboost_churn_model.pkl',
        '/tmp/test_model.pkl'
    )
    s3.download_file(
        BUCKET_NAME,
        'models/scaler.pkl',
        '/tmp/test_scaler.pkl'
    )
    model  = joblib.load('/tmp/test_model.pkl')
    scaler = joblib.load('/tmp/test_scaler.pkl')

    # Load test data
    obj    = s3.get_object(Bucket=BUCKET_NAME, Key='data/features/X_test.csv')
    X_test = pd.read_csv(io.BytesIO(obj['Body'].read()))

    # Make predictions
    X_scaled = scaler.transform(X_test)
    preds    = model.predict_proba(X_scaled)[:, 1]

    # Predictions should be between 0 and 1
    assert all(0 <= p <= 1 for p in preds)
    print(f"✅ Test 3 passed: Model predicts correctly ({len(preds)} predictions)")

# ── Test 4: Model performance is acceptable ───────────────
def test_model_performance():
    s3 = boto3.client('s3')

    # Load metadata
    obj      = s3.get_object(Bucket=BUCKET_NAME, Key='models/latest/metadata.json')
    metadata = json.loads(obj['Body'].read())

    auc = metadata.get('auc', 0)

    # AUC must be above 0.85 to deploy
    assert auc >= 0.85, f"AUC {auc} is below minimum threshold of 0.85!"
    print(f"✅ Test 4 passed: Model AUC {auc} meets minimum threshold")

# ── Test 5: S3 data exists ────────────────────────────────
def test_s3_data_exists():
    s3       = boto3.client('s3')
    required = [
        'data/features/X_train.csv',
        'data/features/X_test.csv',
        'data/features/y_train.csv',
        'data/features/y_test.csv',
        'models/latest/xgboost_churn_model.pkl',
        'models/scaler.pkl',
        'models/latest/metadata.json'
    ]
    for key in required:
        try:
            s3.head_object(Bucket=BUCKET_NAME, Key=key)
            print(f"   ✅ Found: {key}")
        except:
            pytest.fail(f"Missing required file: {key}")
    print("✅ Test 5 passed: All required S3 files exist")

# ── Test 6: Monitoring report exists ──────────────────────
def test_monitoring_report_exists():
    s3  = boto3.client('s3')
    obj = s3.get_object(
        Bucket=BUCKET_NAME,
        Key='monitoring/latest_summary.json'
    )
    report = json.loads(obj['Body'].read())
    assert 'max_psi'    in report
    assert 'checked_at' in report
    print("✅ Test 6 passed: Monitoring report exists and is valid")

if __name__ == "__main__":
    test_model_loads()
    test_scaler_loads()
    test_model_predicts()
    test_model_performance()
    test_s3_data_exists()
    test_monitoring_report_exists()
    print("\\n🎉 All tests passed!")
"""

# Save test file
test_dir  = os.path.expanduser("~/churn-mlops/tests")
test_path = os.path.join(test_dir, "test_pipeline.py")
os.makedirs(test_dir, exist_ok=True)

with open(test_path, 'w') as f:
    f.write(test_code)

print("✅ Test file created!")
print(f"📁 Saved to: {test_path}")

✅ Test file created!
📁 Saved to: /home/sagemaker-user/churn-mlops/tests/test_pipeline.py


In [4]:
# Run all 6 tests right now
# This is exactly what GitHub Actions will do
# before every deployment

import subprocess
import sys

print("🧪 Running all tests...")
print("="*50)

test_path = os.path.expanduser("~/churn-mlops/tests/test_pipeline.py")

result = subprocess.run(
    [sys.executable, test_path],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode == 0:
    print("✅ ALL TESTS PASSED!")
    print("🚀 Code is ready for deployment!")
else:
    print("❌ Some tests failed:")
    print(result.stderr)

🧪 Running all tests...
✅ Test 1 passed: Model loads correctly
✅ Test 2 passed: Scaler loads correctly
✅ Test 3 passed: Model predicts correctly (228 predictions)
✅ Test 4 passed: Model AUC 0.9966 meets minimum threshold
   ✅ Found: data/features/X_train.csv
   ✅ Found: data/features/X_test.csv
   ✅ Found: data/features/y_train.csv
   ✅ Found: data/features/y_test.csv
   ✅ Found: models/latest/xgboost_churn_model.pkl
   ✅ Found: models/scaler.pkl
   ✅ Found: models/latest/metadata.json
✅ Test 5 passed: All required S3 files exist
✅ Test 6 passed: Monitoring report exists and is valid

🎉 All tests passed!

✅ ALL TESTS PASSED!
🚀 Code is ready for deployment!


In [5]:
# GitHub Actions is a robot that runs automatically
# every time you push code to GitHub
# It runs your tests and if they pass - deploys automatically!

github_actions = """
name: MLOps CI/CD Pipeline

on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ main ]

jobs:
  test-and-deploy:
    runs-on: ubuntu-latest

    steps:
    # Step 1: Download your code
    - name: Checkout code
      uses: actions/checkout@v3

    # Step 2: Install Python
    - name: Set up Python
      uses: actions/setup-python@v4
      with:
        python-version: '3.11'

    # Step 3: Install libraries
    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip
        pip install -r requirements.txt

    # Step 4: Run all tests
    - name: Run tests
      env:
        AWS_ACCESS_KEY_ID:     ${{ secrets.AWS_ACCESS_KEY_ID }}
        AWS_SECRET_ACCESS_KEY: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
        AWS_DEFAULT_REGION:    ${{ secrets.AWS_DEFAULT_REGION }}
      run: |
        python tests/test_pipeline.py

    # Step 5: Build Docker container
    - name: Build Docker image
      run: |
        docker build -t churn-mlops-api:latest .

    # Step 6: Login to AWS ECR
    - name: Login to Amazon ECR
      env:
        AWS_ACCESS_KEY_ID:     ${{ secrets.AWS_ACCESS_KEY_ID }}
        AWS_SECRET_ACCESS_KEY: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
        AWS_DEFAULT_REGION:    ${{ secrets.AWS_DEFAULT_REGION }}
      run: |
        aws ecr get-login-password --region us-east-1 | \\
        docker login --username AWS \\
        --password-stdin ${{ secrets.ECR_REGISTRY }}

    # Step 7: Push to ECR
    - name: Push to ECR
      run: |
        docker tag churn-mlops-api:latest \\
        ${{ secrets.ECR_REGISTRY }}/churn-mlops-api:latest
        docker push ${{ secrets.ECR_REGISTRY }}/churn-mlops-api:latest
        echo "✅ Docker image pushed to ECR!"
"""

# Save GitHub Actions workflow
workflow_dir = os.path.expanduser("~/churn-mlops/.github/workflows")
os.makedirs(workflow_dir, exist_ok=True)

workflow_path = os.path.join(workflow_dir, "mlops_pipeline.yml")
with open(workflow_path, 'w') as f:
    f.write(github_actions)

print("✅ GitHub Actions workflow created!")
print(f"📁 Saved to: {workflow_path}")

✅ GitHub Actions workflow created!
📁 Saved to: /home/sagemaker-user/churn-mlops/.github/workflows/mlops_pipeline.yml


In [7]:
readme = (
    "# Netflix Customer Churn Prediction - MLOps Pipeline\n\n"
    "## Project Overview\n"
    "An industry-level MLOps pipeline that predicts Netflix customer churn "
    "using XGBoost with automated retraining, drift monitoring, and CI/CD on AWS.\n\n"
    "## Architecture\n"
    "- Data Storage: AWS S3\n"
    "- Feature Engineering: Pandas + Scikit-learn\n"
    "- Experiment Tracking: MLflow\n"
    "- Model Training: XGBoost + Optuna (Bayesian hyperparameter tuning)\n"
    "- Explainability: SHAP\n"
    "- Model Serving: FastAPI REST API\n"
    "- Monitoring: PSI + KS drift detection\n"
    "- Retraining: Automated drift-triggered retraining\n"
    "- CI/CD: GitHub Actions + Docker + AWS ECR\n\n"
    "## Model Performance\n"
    "- AUC:      0.996\n"
    "- Accuracy: 97%\n"
    "- F1 Score: 97%\n\n"
    "## API Endpoints\n"
    "- GET  /         — API status\n"
    "- GET  /health   — Health check\n"
    "- POST /predict  — Predict churn\n"
    "- GET  /model/info — Model metadata\n\n"
    "## AWS Services Used\n"
    "- S3         — Data and model storage\n"
    "- SageMaker  — Training and serving\n"
    "- ECR        — Docker container registry\n"
    "- GitHub Actions — CI/CD pipeline\n\n"
    "## How to Run\n"
    "1. Upload CSV to S3\n"
    "2. Run notebooks 01 to 08 in order\n"
    "3. API starts automatically\n"
    "4. Monitoring runs on schedule\n"
    "5. Retraining triggers automatically on drift\n"
)

readme_path = os.path.expanduser("~/churn-mlops/README.md")
with open(readme_path, 'w') as f:
    f.write(readme)

print("✅ README created!")
print(f"📁 Saved to: {readme_path}")

✅ README created!
📁 Saved to: /home/sagemaker-user/churn-mlops/README.md


In [8]:
# Print your complete project summary

print("🎉" * 20)
print()
print("   YOUR PROJECT IS 100% COMPLETE!")
print()
print("🎉" * 20)
print()
print("📊 FINAL PROJECT SUMMARY:")
print("="*55)
print()
print("✅ Step 1 — Data cleaned and stored in AWS S3")
print("✅ Step 2 — 6 smart features engineered")
print("✅ Step 3 — XGBoost trained with Optuna (AUC: 0.996)")
print("✅ Step 4 — SHAP explainability charts")
print("✅ Step 5 — FastAPI REST API serving predictions")
print("✅ Step 6 — PSI + KS drift monitoring")
print("✅ Step 7 — Automated retraining pipeline")
print("✅ Step 8 — Docker + GitHub Actions CI/CD")
print()
print("☁️  AWS SERVICES USED:")
print("   ✅ Amazon S3       — data and model storage")
print("   ✅ SageMaker       — training and serving")
print("   ✅ AWS ECR         — Docker registry")
print("   ✅ GitHub Actions  — CI/CD pipeline")
print()
print("🏆 WHAT MAKES THIS INDUSTRY LEVEL:")
print("   ✅ Automated drift detection")
print("   ✅ Self-healing retraining pipeline")
print("   ✅ Model explainability with SHAP")
print("   ✅ Versioned models in S3")
print("   ✅ Clean structured logging")
print("   ✅ Automated tests before deployment")
print("   ✅ Dockerized deployment")
print("   ✅ Professional REST API")
print()
print("🎓 You built a project that most professionals")
print("   with 2-3 years experience cannot build!")
print()
print("🚀 Show this to your mentor — they will be impressed!")

🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉

   YOUR PROJECT IS 100% COMPLETE!

🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉

📊 FINAL PROJECT SUMMARY:

✅ Step 1 — Data cleaned and stored in AWS S3
✅ Step 2 — 6 smart features engineered
✅ Step 3 — XGBoost trained with Optuna (AUC: 0.996)
✅ Step 4 — SHAP explainability charts
✅ Step 5 — FastAPI REST API serving predictions
✅ Step 6 — PSI + KS drift monitoring
✅ Step 7 — Automated retraining pipeline
✅ Step 8 — Docker + GitHub Actions CI/CD

☁️  AWS SERVICES USED:
   ✅ Amazon S3       — data and model storage
   ✅ SageMaker       — training and serving
   ✅ AWS ECR         — Docker registry
   ✅ GitHub Actions  — CI/CD pipeline

🏆 WHAT MAKES THIS INDUSTRY LEVEL:
   ✅ Automated drift detection
   ✅ Self-healing retraining pipeline
   ✅ Model explainability with SHAP
   ✅ Versioned models in S3
   ✅ Clean structured logging
   ✅ Automated tests before deployment
   ✅ Dockerized deployment
   ✅ Professional REST API

🎓 You built a project that most professionals
   with 2-3 years experie

In [1]:
# Run all 6 tests right now
# This is exactly what GitHub Actions will do
# before every deployment

import subprocess
import sys
import os

print("🧪 Running all 6 tests...")
print("="*50)

test_path = os.path.expanduser("~/churn-mlops/tests/test_pipeline.py")

result = subprocess.run(
    [sys.executable, test_path],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode == 0:
    print("="*50)
    print("✅ ALL 6 TESTS PASSED!")
    print("🚀 Code is ready for deployment!")
else:
    print("❌ Some tests failed:")
    print(result.stderr)

🧪 Running all 6 tests...
✅ Test 1 passed: Model loads correctly
✅ Test 2 passed: Scaler loads correctly
✅ Test 3 passed: Model predicts correctly (228 predictions)
✅ Test 4 passed: Model AUC 0.9966 meets minimum threshold
   ✅ Found: data/features/X_train.csv
   ✅ Found: data/features/X_test.csv
   ✅ Found: data/features/y_train.csv
   ✅ Found: data/features/y_test.csv
   ✅ Found: models/latest/xgboost_churn_model.pkl
   ✅ Found: models/scaler.pkl
   ✅ Found: models/latest/metadata.json
✅ Test 5 passed: All required S3 files exist
✅ Test 6 passed: Monitoring report exists and is valid

🎉 All tests passed!

✅ ALL 6 TESTS PASSED!
🚀 Code is ready for deployment!
